# Chapter 09 — LLM 기반 로그 해석 (실습)

> **Chapter 09 · Part II**

Claude API 로 로그를 분류하고, mock 모드로도 끝까지 동작합니다.

> **주의**: 실제 Claude API 호출에는 API key 가 필요하다 (https://console.anthropic.com).
> 키가 있으면 실제 호출, 없으면 규칙 기반 mock 으로 끝까지 진행된다. mock 경로는 결정론이라
> 분류 결과를 재현할 수 있지만, **토큰·캐시 사용량 측정은 실제 호출에서만 가능하다.**

## 실습 목표
1. Claude API 첫 호출 — 응답 본문과 `usage` 토큰·비용 확인
2. 시스템 프롬프트 + JSON 구조화 출력
3. 정답 라벨이 붙은 합성 로그 50 건 일괄 분류 + 혼동 행렬 채점
4. ML 1 차 + LLM 2 차 하이브리드 — 전량 LLM 대비 LLM 호출 수·정확도 비교
5. Prompt caching 실측 — 모델별 최소 캐시 길이 조건과 비용 계산

In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q anthropic
# !pip install -q pandas
# !pip install -q numpy

In [ ]:
import os, json, re, time
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

API_KEY = os.environ.get('ANTHROPIC_API_KEY')  # Colab Secrets 또는 환경변수
USE_REAL_API = bool(API_KEY)
MODEL = 'claude-haiku-4-5'   # 대량 1차 분류용. thinking 기본 꺼짐
PRICE_IN, PRICE_OUT = 1.0, 5.0        # Haiku 4.5 — 입력 $1 / 출력 $5 per MTok (본문 §9.3.1)

if USE_REAL_API:
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)
    print(f'Real API mode — {MODEL}')
else:
    client = None
    print('Mock mode (API key 없음) — 분류는 규칙 기반 mock, 토큰·캐시 측정은 건너뛴다')

---
## Step 0. 첫 호출과 usage

`resp.usage` 는 이 호출이 실제로 쓴 토큰을 알려 준다. 비용 계산도 추정이 아니라 이 값에서 시작한다.
응답 앞에 `thinking` 블록이 올 수 있으므로 `content[0]` 을 가정하지 않고 `text` 블록만 모은다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: def text_of(resp):

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def text_of(resp):
    ...
    return ...

if USE_REAL_API:
    resp = ...
        model = ...
        max_tokens = ...
        messages = ...
                   ...
    u = ...
    cost = ...
    print(f'\nusage: input = ...
else:
    ...
```
</details>


---
## Step 1. 시스템 프롬프트 설정

판정 규칙과 **로그 필드의 의미**를 함께 적는다. 필드 의미를 알려 주지 않으면
`fail_count_5m` 같은 맥락 필드가 있어도 모델이 한 줄만 보고 판단하게 된다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: SYSTEM_PROMPT = ...

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
SYSTEM_PROMPT = ...
...
3. 판정 근거가 부족하면 UNKNOWN, severity = ...
...
print(f'system prompt length = ...
```
</details>


---
## Step 2. 정답 라벨이 붙은 합성 로그 50 건

채점하려면 정답이 있어야 한다. 라벨은 로그 딕셔너리 **밖**의 리스트에 따로 두어
모델 입력에 정답이 섞여 들어가지 않게 한다. Brute 로 라벨한 로그에는 판정 근거가 되는
`fail_count_5m` 이 실제로 들어 있다 — 한 줄만 보고는 알 수 없는 것을 정답으로 삼지 않는다.
정상 30 건에는 `select` 나 `sql-injection` 같은 단어가 URL 에 들어 있는 경계 로그 2 건을 섞었다.
키워드만 보는 탐지기가 오탐하기 쉬운 자리다. 모델이 이 두 건을 어떻게 판정했는지는
혼동 행렬의 BENIGN 행에서 확인한다.

In [ ]:
rng = np.random.default_rng(42)
BENIGN_URLS = ['/index.html', '/products', '/api/v1/users', '/static/app.js', '/health']
ATTACK_URLS = [("/login?id=' OR 1=1--", 'SQLi'),
               ('/item?id=1 UNION SELECT NULL,NULL--', 'SQLi'),
               ("/item?id=1' OR '1'='1", 'SQLi'),
               ('/report?year=2026 UNION SELECT password FROM users--', 'SQLi'),
               ('/search?q=<script>alert(1)</script>', 'XSS'),
               ('/comment?text=<script>document.cookie</script>', 'XSS'),
               ('/profile?name=<script>alert(document.domain)</script>', 'XSS'),
               ('/download?file=../../etc/passwd', 'Other'),
               ('/view?f=../../../windows/win.ini', 'Other'),
               ('/backup?path=../../var/log/auth.log', 'Other')]

# 공격처럼 보이지만 정상인 경계 로그 — 키워드만 보는 탐지기가 오탐하는 자리
TRICKY_BENIGN = ['/search?q=select a plan for 2026', '/blog/how-to-prevent-sql-injection']

logs, labels = [], []
for i in range(28):                                    # 평범한 정상 28
    logs.append({'src': f'10.0.0.{int(rng.integers(1, 50))}', 'method': 'GET',
                 'url': BENIGN_URLS[i % len(BENIGN_URLS)], 'ua': 'Mozilla/5.0',
                 'status': 200, 'fail_count_5m': 0})
    labels.append('BENIGN')
for url in TRICKY_BENIGN:                              # 경계 정상 2
    logs.append({'src': '10.0.0.7', 'method': 'GET', 'url': url,
                 'ua': 'Mozilla/5.0', 'status': 200, 'fail_count_5m': 0})
    labels.append('BENIGN')
for i in range(10):                                    # 페이로드 공격 10 (SQLi 4 · XSS 3 · 경로 조작 3)
    url, lab = ATTACK_URLS[i]
    logs.append({'src': '10.0.0.99', 'method': 'POST', 'url': url,
                 'ua': str(rng.choice(['sqlmap/1.5', 'nikto/2.1', 'curl/7.0'])),
                 'status': int(rng.choice([401, 403, 500])), 'fail_count_5m': 0})
    labels.append(lab)
for i in range(10):                                    # 무차별 대입 10
    logs.append({'src': f'192.168.1.{100 + i}', 'method': 'POST', 'url': '/login',
                 'ua': 'Mozilla/5.0', 'status': 401,
                 'fail_count_5m': int(rng.integers(18, 60))})
    labels.append('Brute')

print(f'총 {len(logs)} 건 — 정답 라벨 분포')
print(pd.Series(labels).value_counts().to_string())
logs[0], labels[0]

---
## Step 3. 분류 함수 — 실제 API 또는 Mock

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 파싱 실패를 BENIGN 으로 돌려주면 정상 로그에서 실패가 정답으로 채점된다

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def mock_classify(log: ...
    ...
    url = ...
    if re.search(r'union\s+select|or\s+1=1', url, re.I):
        return ...
                ...
    if re.search(r'<script>|javascript: ...
        return ...
                ...
    if '../' in url or '%2e%2e%2f' in url.lower():
        return ...
                ...
    if log.get('status') == 401 and log.get('fail_count_5m', 0) >= 10:
        return ...
                ...
    return ...
            ...

def real_classify(log: ...
    resp = ...
        model = ...
        max_tokens = ...
        system = ...
        messages = ...
    ...
    text = ...
    usage = ...
    try:
        return ...
    except json.JSONDecodeError:
        m = ...
        if m:
            try:
                return ...
            except json.JSONDecodeError:
                pass
        # 파싱 실패를 BENIGN 으로 돌려주면 정상 로그에서 실패가 정답으로 채점된다
        return ...

classify = ...
```
</details>


In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 50 건 일괄 분류 (실제 API 는 병렬)

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
# 50 건 일괄 분류 (실제 API 는 병렬)
start = ...
if USE_REAL_API:
    with ThreadPoolExecutor(max_workers=5) as ex:
        results = ...
else:
    results = ...
elapsed = ...

df = ...
                    ...
                   for lg, t, r in zip(logs, labels, results)])
acc = ...
tok = ...
...
if USE_REAL_API:
    tin = ...
    ...

...

if not USE_REAL_API:
    ...

wrong = ...
...
print(wrong[['url', 'ua', 'status', 'fail_count_5m', 'truth', 'pred']].to_string(index = ...
      if len(wrong) else '없음')
```
</details>


---
## Step 4. ML 1 차 + LLM 2 차 하이브리드

ML 1 차는 명백한 정상·명백한 공격을 즉시 처리하고, 모호한 구간만 LLM 에 넘긴다.
여기서 확인할 것은 두 가지다 — **LLM 호출을 얼마나 줄이는가**, 그리고 **그 대가로 정확도를 얼마나 잃는가**.
ML 은 카테고리를 구분하지 못하므로 비교는 `공격 / 정상` 이진 판정으로 맞춘다.
Step 3 에서 이미 분류한 결과를 재사용하므로 같은 로그를 다시 호출하지 않는다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: '../' 를 정규식에 넣을 때 점을 이스케이프하지 않으면 '.' 가 임의 문자가 되어

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def ml_score(log: ...
    ...
    url, ua = ...
    # '../' 를 정규식에 넣을 때 점을 이스케이프하지 않으면 '.' 가 임의 문자가 되어
    # /api/v1/users 의 'pi/' 까지 걸린다. 정상 로그가 즉시 차단으로 튀는 오탐이다.
    if re.search(r'union\s+select|or\s+1=1|<script>|javascript: ...
        return ...
    if re.search(r'sqlmap|nikto', ua, re.I):
        return ...
    if log.get('status') == 401:
        return ...
    return ...

fp = ...
...
```
</details>


In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: def to_binary(cat):

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def to_binary(cat):
    if cat is None:
        return ...
    if cat == 'UNKNOWN':
        return ...
    if cat in {'SQLi', 'XSS', 'DDoS', 'PortScan', 'Brute', 'Other'}:
        return ...
    return ...

rows = ...
for lg, truth, llm in zip(logs, labels, results): ...
    s = ...
    if s < 0.3:
        route, pred, calls = ...
    elif s > 0.7:
        route, pred, calls = ...
    else:
        route, pred, calls = ...
    ...
routing = ...

...

cmp = ...
    ...
cmp['호출 절감'] = ...
cmp['이진 정확도'] = ...
...
print(cmp.to_string(index = ...

diff = ...
...
print(diff[['url', 'ml', 'route', 'truth_bin', 'llm_only_pred', 'hybrid_pred']].to_string(index = ...
      if len(diff) else '없음 (ML 1차가 거른 구간에서 LLM 과 같은 판정)')
```
</details>


---
## Step 5. Prompt caching 실측

`cache_control` 을 달았다고 캐시가 걸리는 것은 아니다. **캐시 가능한 프리픽스에는 모델별 최소
토큰 수가 있고, 그보다 짧으면 오류 없이 조용히 캐시되지 않는다.**

| 모델 | 최소 캐시 길이 |
|---|---|
| Opus 5 | 512 토큰 |
| Sonnet 5 | 1,024 토큰 |
| Haiku 4.5 | 4,096 토큰 |

(출처: Anthropic, "Prompt caching" — https://platform.claude.com/docs/en/build-with-claude/prompt-caching)

그래서 Step 1 의 300 자짜리 시스템 프롬프트에 `cache_control` 을 달면 Haiku 에서는 아무 일도
일어나지 않는다. 아래 셀은 **짧은 프롬프트**와 **최소 길이를 넘긴 긴 프롬프트**를 각각 두 번씩
호출해 `usage.cache_creation_input_tokens` / `cache_read_input_tokens` 로 직접 확인한다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 최소 길이를 넘기는 고정 프리픽스 — 실무에서는 사내 자산 인벤토리·탐지 규칙표처럼

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
# 최소 길이를 넘기는 고정 프리픽스 — 실무에서는 사내 자산 인벤토리·탐지 규칙표처럼
# 요청마다 바뀌지 않는 참조 자료가 이 자리에 온다.
def asset_appendix(n_rows: ...
    lines = ...
    for i in range(n_rows):
        ...
                     f'owner = ...
                     f'criticality = ...
    return ...

MIN_CACHE_TOKENS = ...
MIN_FOR_MODEL = ...
SAMPLE_LOG = ...

def count_tokens(system_blocks):
    r = ...
        model = ...
        messages = ...
    return ...

def cached_system(text):
    return ...

def call_once(system_blocks):
    resp = ...
        model = ...
        messages = ...
    u = ...
    return ...
            ...

if USE_REAL_API:
    # 최소 길이를 넘을 때까지 프리픽스를 늘린다 (count_tokens 는 과금되지 않는다)
    n = ...
    while True:
        long_system = ...
        n_tok = ...
        if n_tok > MIN_FOR_MODEL * 1.1 or n > 2000:
            break
        n = ...
    short_tok = ...
    print(f'짧은 프롬프트 = ...
    print(f'긴 프롬프트 = ...

    usage_rows = ...
    for label, sysmsg in [('짧은 프롬프트', SYSTEM_PROMPT), ('긴 프롬프트', long_system)]:
        for attempt in ('1회차(기록)', '2회차(재사용)'):
            u = ...
            ...
    usage_df = ...
    print(usage_df.to_string(index = ...
    ...
else:
    usage_df = ...
    ...
```
</details>


In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 비용 = 입력 × $1 + 캐시 기록 × $1×1.25 + 캐시 읽기 × $1×0.1 + 출력 × $5 (per MTok, 본문 §9.3.1·§9.4)

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
# 비용 = 입력 × $1 + 캐시 기록 × $1×1.25 + 캐시 읽기 × $1×0.1 + 출력 × $5 (per MTok, 본문 §9.3.1·§9.4)
def call_cost(u):
    return ...
            ...

if USE_REAL_API:
    cost_df = ...
    cost_df['호출 비용($)'] = ...
    ...
          .to_string(index = ...

    ...
    reuse = ...
    write = ...
    no_cache = ...
                ...
else:
    ...
```
</details>


---
## 정리
- LLM 한 번의 호출로 보안 로그의 분류·설명·대응 추천을 함께 받았습니다. `usage` 로 호출마다 쓴 토큰과 비용을 확인했습니다.
- 정답 라벨을 붙여 두면 일괄 분류 결과를 혼동 행렬로 채점할 수 있습니다. 어느 카테고리에서 틀리는지가 프롬프트를 고치는 출발점입니다.
- ML 이 명백한 케이스를 먼저 거르는 하이브리드는 LLM 호출 수를 크게 줄입니다. 호출 절감과 이진 정확도를 같은 표에서 비교했습니다.
- ML 1 차의 정규식은 자기 자신이 오탐 원인이 될 수 있습니다. `../` 의 점을 이스케이프하지 않으면 `/api/v1/users` 같은 정상 경로가 즉시 차단으로 튀므로, 정상 로그에 대한 회귀 점검을 넣었습니다.
- Prompt caching 은 프리픽스가 모델별 최소 길이(Haiku 4.5 는 4,096 토큰) 를 넘을 때만 걸리고, 짧으면 오류 없이 무시됩니다. `cache_creation_input_tokens`·`cache_read_input_tokens` 로 직접 확인해야 합니다.

> **다음 챕터:** Prompt Engineering 심화와 XAI — Role·Few-shot·CoT·구조화 출력 패턴.